In [ ]:
from imutils import paths
import numpy as np
import argparse
import imutils
import pickle
import cv2
import os

# Recognize faces with OpenCV

In [ ]:
from keras.models import load_model

Loading all the Face Detector Models 

In [ ]:
protoPath = "/content/drive/MyDrive/Face_reco/Face Recognition from Video/deploy.prototxt.txt"

modelPath = "/content/drive/MyDrive/Face_reco/Face Recognition from Video/res10_300x300_ssd_iter_140000.caffemodel"

detector = cv2.dnn.readNetFromCaffe(protoPath, modelPath)

embedder = cv2.dnn.readNetFromTorch("/content/drive/MyDrive/Face_reco/Face Recognition from Video/openface.nn4.small2.v1.t7")

#Load model file
recognizer = load_model('/content/drive/MyDrive/Face_reco/Face Recognition from Video/recognizer.h5')

le = pickle.loads(open("/content/drive/MyDrive/Face_reco/Face Recognition from Video/label_Encoder", "rb").read())
le.classes_[3] = 'Soni'

error: ignored

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
def image_recognize(image):
    image = cv2.resize(image, (640, 480))
    (h, w) = image.shape[:2]
    imageBlob = cv2.dnn.blobFromImage( cv2.resize(image, (300, 300)), 1.0, (300, 300), (104.0, 177.0, 123.0), 
                                      swapRB=False, crop=False)
    
    detector.setInput(imageBlob)
    detections = detector.forward()
    
    for i in range(0, detections.shape[2]):
    
        # extract the confidence (i.e., probability) associated with the prediction
        confidence = detections[0, 0, i, 2]

        # filter out weak detections
        if confidence > 0.85:
            # compute the (x, y)-coordinates of the bounding box for the face

            box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
            (startX, startY, endX, endY) = box.astype("int")

            # extract the face ROI

            face = image[startY:endY, startX:endX]
            (fH, fW) = face.shape[:2]

            # ensure the face width and height are sufficiently large
    
            if fW < 30 or fH < 30:
                continue
                
            faceBlob = cv2.dnn.blobFromImage(face, 1.0 / 255, (96, 96), (0, 0, 0), swapRB=True, crop=False)
            embedder.setInput(faceBlob)
            vec = embedder.forward()

            # perform classification to recognize the face
            
            preds = recognizer.predict(vec)[0]
            j = np.argmax(preds)

            proba = preds[j]
            name = le.classes_[j]
            if proba >= 0.40:
                text = name

                cv2.rectangle(image, (startX, startY), (endX, endY), (0, 255, 0), 2)
                cv2.putText(image, text, (startX, startY), cv2.FONT_HERSHEY_DUPLEX, 1.2, (0, 255, 0), 2)

    return np.array(image)

In [ ]:
import matplotlib.pyplot as plt

# Recognize faces in video streams

In [ ]:
# using a webcam
cam = cv2.VideoCapture("/content/drive/MyDrive/Face_reco/Face Recognition from Video/Crowd - 6582.mp4")

frame_width = int(cam.get(3))
frame_height = int(cam.get(4))

out = cv2.VideoWriter('/content/drive/MyDrive/Face_reco/Face Recognition from Video/Face-Mask.mp4',cv2.VideoWriter_fourcc('M','J','P','G'), 10, (frame_height,frame_width))
#cam.set(3,640)
#cam.set(4,480)
#cam.set(10,130)


while True:
    success, image = cam.read()
    if success == True:
        f_image = image_recognize(image)
        #print(image)
        #cv2.imshow("Result", image)
        out.write(f_image)
        #print(out)

    else:
        break 
#print(out)
#cam.release()
out.release()
#cv2.destroyAllWindows()

In [ ]:
cam = cv2.VideoCapture("/content/drive/MyDrive/Face_reco/Face Recognition from Video/Face-Mask.mp4")
#while True:
    #success, image = cam.read()
    #if success == True:
      #image1 = cv2.imread(image)
      #plt.imshow(image1))
image = cam.read() 
print(cam)


<VideoCapture 0x7fa30402c1f0>
